In [1]:
import os
import math

import torch
import torch.nn as nn
from tokenizers import Tokenizer                     # 分词工具
from torchtext.vocab import build_vocab_from_iterator    # 构建词典
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.nn.functional import pad, log_softmax   # pad用于文本对齐
from transformers import AutoTokenizer

import matplotlib.pyplot as plt
import numpy as np

# ========== 选择设备（优先GPU，无则用CPU） ==========
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"当前使用设备: {device}")  # 输出应显示cuda:0（GPU）或cpu

# 加载基础的分词器模型，使用的是基础的bert模型。`uncased`意思是不区分大小写
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

# 分词封装
def en_tokenizer(line):
    """
    定义英文分词器
    :param line: 一句英文句子，例如"I'm learning Deep learning."
    :return: subword分词后的结果，例如：['i', "'", 'm', 'learning', 'deep', 'learning', '.']
    """
    # 使用bert进行分词，直接获取tokens
    return tokenizer.tokenize(line)

# 交换源语言和目标语言文件路径
zh_filepath = r"D:\homework\homework\Summer\files\train.zh"  # 源语言：中文
en_filepath = r"D:\homework\homework\Summer\files\train.en"  # 目标语言：英文


当前使用设备: cuda:0


D:\anaconder\envs\trans_env\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
# 中文作为源语言，生成源语言词汇表
def yield_zh_tokens():
    """每次yield一个分词后的中文句子"""
    with open(zh_filepath, encoding='utf-8') as fd:
        for line in fd:
            yield zh_tokenizer(line)

def zh_tokenizer(line):
    """
    定义中文分词器
    :param line: 中文句子，例如：机器学习
    :return: 分词结果，例如['机','器','学','习']
    """
    return list(line.strip().replace(" ", ""))

zh_tok = yield_zh_tokens()
for t in  zh_tok:
    print(t)
    break

# 源语言词汇表路径
src_vocab_file = "D:/homework/homework/Summer/files/vocab_zh.pt"

# 英文作为目标语言，生成目标语言词汇表
def yield_en_tokens():
    """每次yield一个分词后的英文句子"""
    with open(en_filepath, encoding='utf-8') as fd:
        for line in fd:
            yield en_tokenizer(line)

en_tok = yield_en_tokens()
for t in  en_tok:
    print(t)
    break

# 目标语言词汇表路径
tgt_vocab_file = "D:/homework/homework/Summer/files/vocab_en.pt"

# -------------------------- 加载已生成的词典 --------------------------
# 加载源语言词典
src_vocab = torch.load(src_vocab_file)
print(f"✅ 成功加载中文词典，词典大小：{len(src_vocab)}")

# 加载目标语言词典
tgt_vocab = torch.load(tgt_vocab_file)
print(f"✅ 成功加载英文词典，词典大小：{len(tgt_vocab)}")
# ----------------------------------------------------------------------------

# 打印看一下效果
print("中文词典大小:", len(src_vocab))
print(dict((i, src_vocab.lookup_token(i)) for i in range(10)))


['一', '对', '丹', '顶', '鹤', '正', '监', '视', '着', '它', '们', '的', '筑', '巢', '领', '地']
['a', 'pair', 'of', 'red', '-', 'crowned', 'cranes', 'have', 'stake', '##d', 'out', 'their', 'nesting', 'territory']
✅ 成功加载中文词典，词典大小：8280
✅ 成功加载英文词典，词典大小：27584
中文词典大小: 8280
{0: '<s>', 1: '</s>', 2: '<pad>', 3: '<unk>', 4: '。', 5: '的', 6: '，', 7: '我', 8: '你', 9: '是'}


In [3]:
class TranslationDataset(Dataset):

    def __init__(self):
        # 加载源语言（中文）tokens和目标语言（英文）tokens
        self.src_tokens = self.load_tokens(zh_filepath, zh_tokenizer, src_vocab, 'zh')  # 中文作为源语言
        self.tgt_tokens = self.load_tokens(en_filepath, en_tokenizer, tgt_vocab, 'en')  # 英文作为目标语言

        self.row_count = len(self.src_tokens)

    def __getitem__(self, index):
        return self.src_tokens[index], self.tgt_tokens[index]

    def __len__(self):
        return self.row_count

    def load_tokens(self, file, tokenizer, vocab, lang):
        """
        加载tokens，即将文本句子们转换成index们。
        :param file: 文件路径
        :param tokenizer: 分词器
        :param vocab: 词典
        :param lang: 语言
        :return: 构造好的tokens列表
        """
        tokens_list = []
        with open(file, encoding='utf-8') as fd:
            for line in fd:
                tokens = tokenizer(line) 
                tokens = vocab(tokens)    # 将文本分词结果通过词典转成index
                tokens_list.append(tokens)
        return tokens_list

ds = TranslationDataset()
print(ds[0])


([12, 40, 1173, 1084, 3169, 164, 693, 397, 84, 100, 14, 5, 1218, 2398, 535, 67], [11, 2731, 12, 554, 19, 17230, 18104, 27, 3081, 203, 57, 102, 18856, 3653])


In [4]:
max_length = 72
def collate_fn(batch):
    """
    将dataset的数据进一步处理，并组成一个batch。
    :param batch: 一个batch的数据
    :return: 填充后的且等长的数据，包括src, tgt, tgt_y, n_tokens
    """
    # 定义'<bos>'的index，在词典中为0
    bs_id = torch.tensor([0])
    # 定义'<eos>'的index
    eos_id = torch.tensor([1])
    # 定义<pad>的index
    pad_id = 2
    # 用于存储处理后的src=zh和tgt=en
    src_list, tgt_list = [], []

    # 循环遍历句子对儿
    for (_src, _tgt) in batch:
        """
        _src: 中文句子对应的index
        _tgt: 英文句子对应的index
        """
        # 将<bos>，句子index和<eos>拼到一块
        processed_src = torch.cat([bs_id, torch.tensor(_src, dtype=torch.int64), eos_id], dim=0)
        processed_tgt = torch.cat([bs_id, torch.tensor(_tgt, dtype=torch.int64), eos_id], dim=0)

        # 填充到max_length长度
        processed_src = pad(processed_src, (0, max_length - len(processed_src)), value=pad_id)
        src_list.append(processed_src)
        processed_tgt = pad(processed_tgt, (0, max_length - len(processed_tgt)), value=pad_id)
        tgt_list.append(processed_tgt)

    # 将多个src句子堆叠到一起
    src = torch.stack(src_list)
    tgt = torch.stack(tgt_list)

    # tgt_y是目标句子去掉第一个token，即去掉<bos>
    tgt_y = tgt[:, 1:]
    # tgt是目标句子去掉最后一个token
    tgt = tgt[:, :-1]

    # 计算本次batch要预测的token数
    n_tokens = (tgt_y != 2).sum()

    # 返回batch后的结果
    return src, tgt, tgt_y, n_tokens

train_loader = DataLoader(ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
for src, tgt, tgt_y, n_tokens in train_loader:
    print(src.shape, tgt.shape, tgt_y.shape, n_tokens)
    break


torch.Size([64, 72]) torch.Size([64, 71]) torch.Size([64, 71]) tensor(838)


In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        # 初始化Shape为(max_len, d_model)的PE (positional encoding)
        pe = torch.zeros(max_len, d_model)
        # 初始化一个tensor [[0, 1, 2, 3, ...]]
        position = torch.arange(0, max_len).unsqueeze(1)
        # 这里就是sin和cos括号中的内容，通过e和ln进行了变换
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model)
        )
        # 计算PE(pos, 2i)
        pe[:, 0::2] = torch.sin(position * div_term)
        # 计算PE(pos, 2i+1)
        pe[:, 1::2] = torch.cos(position * div_term)
        # 为了方便计算，在最外面在unsqueeze出一个batch
        pe = pe.unsqueeze(0)
        # 注册为非参数缓冲区
        self.register_buffer("pe", pe)

    def forward(self, x):
        """
        x 为embedding后的inputs，例如(1,7, 128)，batch size为1,7个单词，单词维度为128
        """
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)
        return self.dropout(x)


# --------------------------
# 模型定义
# --------------------------
class TranslationModel(nn.Module):
    def __init__(self, d_model, src_vocab, tgt_vocab, dropout=0.1):
        super(TranslationModel, self).__init__()
        self.src_embedding = nn.Embedding(len(src_vocab), d_model, padding_idx=2)  # 中文嵌入
        self.tgt_embedding = nn.Embedding(len(tgt_vocab), d_model, padding_idx=2)  # 英文嵌入
        self.positional_encoding = PositionalEncoding(d_model, dropout, max_len=max_length)
        self.transformer = nn.Transformer(
            d_model, 
            dropout=dropout, 
            batch_first=True, 
            nhead=8, 
            num_encoder_layers=2, 
            num_decoder_layers=2, 
            dim_feedforward=128)
        self.predictor = nn.Linear(d_model, len(tgt_vocab))

    def forward(self, src, tgt):
        """原有训练用forward：未改动"""
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size()[-1]).to(src.device)
        src_key_padding_mask = TranslationModel.get_key_padding_mask(src).float().to(src.device)
        tgt_key_padding_mask = TranslationModel.get_key_padding_mask(tgt).float().to(src.device)

        src = self.src_embedding(src)
        tgt = self.tgt_embedding(tgt)
        src = self.positional_encoding(src)
        tgt = self.positional_encoding(tgt)
        
        out = self.transformer(
            src, tgt,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )
        return out

    @staticmethod
    def get_key_padding_mask(tokens):
        return tokens == 2

    # 中译英生成函数（调整输出处理）
    def generate(self, src, max_gen_len=72):
        """
        单段文本的自回归生成
        :param src: 单段输入张量（中文），shape=(1, seq_len)
        :return: 生成的英文token列表（不含<bos>）
        """
        self.eval()  # 推理模式
        with torch.no_grad():
            # 1. 编码器处理src（中文）
            src_key_padding_mask = self.get_key_padding_mask(src).float().to(src.device)
            src_emb = self.src_embedding(src)
            src_pe = self.positional_encoding(src_emb)
            memory = self.transformer.encoder(src_pe, src_key_padding_mask=src_key_padding_mask)
            
            # 2. 初始化解码器输入（仅含<bos>）
            tgt_pred = torch.tensor([[0]], dtype=torch.int64).to(src.device)  # <bos>的索引为0
            
            # 3. 自回归生成
            for _ in range(max_gen_len):
                # 生成解码器掩码
                tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_pred.size(1)).to(src.device)
                tgt_key_padding_mask = self.get_key_padding_mask(tgt_pred).float().to(src.device)
                
                # 解码器前向计算
                tgt_emb = self.tgt_embedding(tgt_pred)
                tgt_pe = self.positional_encoding(tgt_emb)
                out = self.transformer.decoder(
                    tgt_pe, memory,
                    tgt_mask=tgt_mask,
                    tgt_key_padding_mask=tgt_key_padding_mask,
                    memory_key_padding_mask=src_key_padding_mask
                )
                
                # 预测下一个token
                logits = self.predictor(out[:, -1, :])
                next_token = sample_top_k(logits, k=5)  # 需要确保sample_top_k函数已定义
                tgt_pred = torch.cat([tgt_pred, next_token], dim=1)
                
                # 遇到<eos>停止
                if next_token.item() == 1:  # <eos>的索引为1
                    break
            
            # 4. 处理输出：去掉<bos>和<eos>
            tgt_pred = tgt_pred.squeeze(0).tolist()[1:]  # 去掉batch维度和<bos>
            if 1 in tgt_pred:  # 去掉<eos>及之后的内容
                tgt_pred = tgt_pred[:tgt_pred.index(1)]
            return tgt_pred


In [13]:
# 初始化中译英模型
model = TranslationModel(200, src_vocab, tgt_vocab)  # src_vocab为中文，tgt_vocab为英文
# 模型迁移到GPU
model = model.to(device)

# 验证模型输出形状
for src, tgt, tgt_y, n_tokens in train_loader:
    # 数据迁移到GPU
    src = src.to(device)
    tgt = tgt.to(device)
    y = model(src, tgt)
    print(y.shape)
    break

optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

class TranslationLoss(nn.Module):
    def __init__(self):
        super(TranslationLoss, self).__init__()
        self.criterion = nn.KLDivLoss(reduction="sum")
        self.padding_idx = 2  # <pad>的索引

    def forward(self, x, target):
        # 1. 对模型输出做log_softmax（保持在原设备）
        x = log_softmax(x, dim=-1)
        
        # 2. 创建true_dist时强制指定与x相同的设备（GPU）
        true_dist = torch.zeros(x.size(), device=x.device)  
        
        # 3. 用target的索引填充true_dist
        true_dist.scatter_(1, target.data.unsqueeze(1), 1.0)
        
        # 4. 处理<pad>位置
        mask = torch.nonzero(target.data == self.padding_idx)
        if mask.dim() > 0:
            true_dist.index_fill_(0, mask.squeeze(), 0.0)
        
        # 5. 计算损失
        return self.criterion(x, true_dist.clone().detach())


torch.Size([64, 71, 200])


In [14]:
# sample_top_k函数
def sample_top_k(logits, k=5):
    """从top k个预测中采样下一个token"""
    logits = logits.squeeze(0)
    top_k_values, top_k_indices = torch.topk(logits, k)
    probabilities = torch.softmax(top_k_values, dim=-1)
    selected_index = torch.multinomial(probabilities, 1)
    return top_k_indices[selected_index].unsqueeze(0)


# --------------------------
# 中译英推理工具
# --------------------------
def multi_seg_tokenize(src_multi, src_vocab, max_length):
    """
    将多段中文文本转换为单段样本列表
    :param src_multi: 多段中文文本，如["我爱你", "他很高兴"]
    :param src_vocab: 中文词典
    :param max_length: 最大序列长度
    :return: 单段样本列表
    """
    single_samples = []
    for seg in src_multi:
        # 中文分词（单字拆分）
        tokens = zh_tokenizer(seg)
        # 转为索引（未知词用<unk>，假设<unk>索引为3）
        seg_idx = [src_vocab.get(token, 3) for token in tokens]
        # 截断超长文本（预留<bos>和<eos>的位置）
        if len(seg_idx) > max_length - 2:
            seg_idx = seg_idx[:max_length - 2]
        # 组成样本（tgt用占位符）
        single_samples.append( (seg_idx, [0]) )
    return single_samples


def translate_multi_seg(src_multi, model, src_vocab, tgt_vocab, max_length, device):
    """
    多段中文文本翻译为英文
    :param src_multi: 多段中文输入文本列表
    :return: 多段英文翻译结果列表
    """
    # 1. 多段转单段样本
    single_samples = multi_seg_tokenize(src_multi, src_vocab, max_length)
    # 2. 批量处理
    loader = DataLoader(
        single_samples,
        batch_size=len(single_samples),  # 一次处理所有段
        collate_fn=collate_fn
    )
    # 3. 模型推理
    model.eval()
    translations = []
    with torch.no_grad():
        for src, _, _, _ in loader:  # 只需要src
            src = src.to(device)
            # 逐段生成翻译
            for i in range(src.shape[0]):
                single_src = src[i].unsqueeze(0)  # 单段输入，shape=(1, seq_len)
                pred_idx = model.generate(single_src, max_gen_len=max_length)
                # 英文单词用空格连接
                pred_text = " ".join([tgt_vocab.lookup_token(idx) for idx in pred_idx 
                                    if idx != 2])  # 过滤<pad>
                translations.append(pred_text)
    return translations



In [15]:
criteria = TranslationLoss()

# 初始化训练参数
epochs = 9  # 总训练轮次
start_epoch = 0  # 起始轮次
resume_training = True  # 是否续训
checkpoint_path = "D:/homework/homework/Summer/files/zh2en_model_best.pt"  # 中译英检查点
best_loss = float('inf')
best_model_path = "D:/homework/homework/Summer/files/zh2en_model_best.pt"  # 中译英最优模型
save_after_step = 100  
step = 0 

# 初始化损失记录
train_step_losses = []  # 每一步损失
train_epoch_avg_losses = []  # 每一轮平均损失


if resume_training:
    try:
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch']
        best_loss = checkpoint['best_loss']
        
        # 加载损失记录
        train_step_losses = torch.load('zh2en_step_losses.pt', map_location=device) if os.path.exists('zh2en_step_losses.pt') else []
        train_epoch_avg_losses = torch.load('zh2en_epoch_losses.pt', map_location=device) if os.path.exists('zh2en_epoch_losses.pt') else []
        
        if start_epoch+1 >= epochs:
            print(f"已完成所有{epochs}轮训练，无需继续")
        else:
            print(f"已加载检查点，从轮次 {start_epoch + 1}/{epochs} 继续训练")
            model.train()
    except:
        print("未找到检查点，将从头开始训练")


for epoch in range(start_epoch + 1, epochs):
    epoch_total_loss = 0.0
    epoch_batch_count = 0
    
    for index, data in enumerate(train_loader):
        src, tgt, tgt_y, n_tokens = data
        src = src.to(device)
        tgt = tgt.to(device)
        tgt_y = tgt_y.to(device)
        
        optimizer.zero_grad()
        out = model(src, tgt)
        out = model.predictor(out)
        loss = criteria(out.contiguous().view(-1, out.size(-1)), tgt_y.contiguous().view(-1)) / n_tokens
        
        # 损失记录与反向传播
        train_step_losses.append(loss.item())
        epoch_total_loss += loss.item()
        epoch_batch_count += 1
        loss.backward()
        optimizer.step()
        
        # 中间日志
        step += 1
        if step % save_after_step == 0:
            print(f"步数：{step:04d}，当前步损失：{loss.detach().item():.4f}")        
    
    # 轮次结束处理
    epoch_avg_loss = epoch_total_loss / epoch_batch_count
    train_epoch_avg_losses.append(epoch_avg_loss)
    print(f"="*50)
    print(f"轮次：{epoch + 1}/{epochs}，本轮平均损失：{epoch_avg_loss:.4f}")
    print(f"当前最优损失：{best_loss:.4f}")

    # 保存损失记录
    torch.save(train_step_losses, 'zh2en_step_losses.pt')
    torch.save(train_epoch_avg_losses, 'zh2en_epoch_losses.pt')

    # 保存最优模型
    if epoch_avg_loss < best_loss:
        best_loss = epoch_avg_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_loss": best_loss
        }, best_model_path)
        print(f"✅ 轮次{epoch + 1}：更新最优模型，保存至 {best_model_path}")
    print(f"="*50 + "\n")


print(f"\n训练结束！")
print(f"最优模型已保存至：{best_model_path}")
print(f"最优损失：{best_loss:.4f}（对应轮次：{train_epoch_avg_losses.index(best_loss)+1}）")


已加载检查点，从轮次 8/10 继续训练
步数：0100，当前步损失：2.3624
步数：0200，当前步损失：2.4205
步数：0300，当前步损失：2.6940
步数：0400，当前步损失：2.5856
步数：0500，当前步损失：2.4442
步数：0600，当前步损失：2.3003
步数：0700，当前步损失：2.6841
步数：0800，当前步损失：2.4175
步数：0900，当前步损失：2.3630
步数：1000，当前步损失：2.7253
步数：1100，当前步损失：2.3340
步数：1200，当前步损失：2.5317
步数：1300，当前步损失：2.5917
步数：1400，当前步损失：2.5836
步数：1500，当前步损失：2.5328
步数：1600，当前步损失：2.5144
步数：1700，当前步损失：2.5059
步数：1800，当前步损失：2.5506
步数：1900，当前步损失：2.4726
步数：2000，当前步损失：2.4127
步数：2100，当前步损失：2.3329
步数：2200，当前步损失：2.5502
步数：2300，当前步损失：2.5160
步数：2400，当前步损失：2.5260
步数：2500，当前步损失：2.4639
步数：2600，当前步损失：2.4506
步数：2700，当前步损失：2.2996
步数：2800，当前步损失：2.7946
步数：2900，当前步损失：2.4510
步数：3000，当前步损失：2.7608
步数：3100，当前步损失：2.5913
步数：3200，当前步损失：2.8815
步数：3300，当前步损失：2.5550
步数：3400，当前步损失：2.7421
步数：3500，当前步损失：2.7674
步数：3600，当前步损失：2.8826
步数：3700，当前步损失：2.6173
步数：3800，当前步损失：2.5832
步数：3900，当前步损失：2.1925
步数：4000，当前步损失：2.6276
步数：4100，当前步损失：2.2215
步数：4200，当前步损失：2.2604
步数：4300，当前步损失：2.6391
步数：4400，当前步损失：2.5523
步数：4500，当前步损失：2.3803
步数：4600，当前步损失：2.5022
步数：4700，当前步损失

KeyboardInterrupt: 

轮次：2/8，本轮平均损失：3.1758
当前最优损失：inf
✅ 轮次2：更新最优模型

轮次：3/8，本轮平均损失：2.7617
当前最优损失：3.1758
✅ 轮次3：更新最优模型，

轮次：4/8，本轮平均损失：2.6620
当前最优损失：2.7617
✅ 轮次4：更新最优模型

5/8 2.5xxx


轮次：7/8，本轮平均损失：2.5573
当前最优损失：2.5787
✅ 轮次7：更新最优模型，

轮次：8/8，本轮平均损失：2.5420
当前最优损失：2.5573

In [17]:
import os
import math
import torch
import torch.nn as nn
from torch.nn.functional import pad, log_softmax
from transformers import AutoTokenizer
from torchtext.vocab import Vocab

# ========== 选择设备（优先GPU，无则用CPU） ==========
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"当前使用设备: {device}")

# 加载英文分词器
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

# 分词函数定义
def en_tokenizer(line):
    return tokenizer.tokenize(line)

def zh_tokenizer(line):
    return list(line.strip().replace(" ", ""))

# 位置编码类定义
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)
        return self.dropout(x)

# 翻译模型类定义
class TranslationModel(nn.Module):
    def __init__(self, d_model, src_vocab, tgt_vocab, dropout=0.1):
        super(TranslationModel, self).__init__()
        self.src_embedding = nn.Embedding(len(src_vocab), d_model, padding_idx=2)
        self.tgt_embedding = nn.Embedding(len(tgt_vocab), d_model, padding_idx=2)
        self.positional_encoding = PositionalEncoding(d_model, dropout, max_len=72)
        self.transformer = nn.Transformer(
            d_model, 
            dropout=dropout, 
            batch_first=True, 
            nhead=8, 
            num_encoder_layers=2, 
            num_decoder_layers=2, 
            dim_feedforward=128)
        self.predictor = nn.Linear(d_model, len(tgt_vocab))

    def forward(self, src, tgt):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size()[-1]).to(src.device)
        src_key_padding_mask = TranslationModel.get_key_padding_mask(src).float().to(src.device)
        tgt_key_padding_mask = TranslationModel.get_key_padding_mask(tgt).float().to(src.device)

        src = self.src_embedding(src)
        tgt = self.tgt_embedding(tgt)
        src = self.positional_encoding(src)
        tgt = self.positional_encoding(tgt)
        
        out = self.transformer(
            src, tgt,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )
        return out

    @staticmethod
    def get_key_padding_mask(tokens):
        return tokens == 2

    def generate(self, src, max_gen_len=72):
        self.eval()
        with torch.no_grad():
            src_key_padding_mask = self.get_key_padding_mask(src).float().to(src.device)
            src_emb = self.src_embedding(src)
            src_pe = self.positional_encoding(src_emb)
            memory = self.transformer.encoder(src_pe, src_key_padding_mask=src_key_padding_mask)
            
            tgt_pred = torch.tensor([[0]], dtype=torch.int64).to(src.device)
            
            for _ in range(max_gen_len):
                tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_pred.size(1)).to(src.device)
                tgt_key_padding_mask = self.get_key_padding_mask(tgt_pred).float().to(src.device)
                
                tgt_emb = self.tgt_embedding(tgt_pred)
                tgt_pe = self.positional_encoding(tgt_emb)
                out = self.transformer.decoder(
                    tgt_pe, memory,
                    tgt_mask=tgt_mask,
                    tgt_key_padding_mask=tgt_key_padding_mask,
                    memory_key_padding_mask=src_key_padding_mask
                )
                
                logits = self.predictor(out[:, -1, :])
                next_token = sample_top_k(logits, k=5)
                tgt_pred = torch.cat([tgt_pred, next_token], dim=1)
                
                if next_token.item() == 1:
                    break
            
            tgt_pred = tgt_pred.squeeze(0).tolist()[1:]
            if 1 in tgt_pred:
                tgt_pred = tgt_pred[:tgt_pred.index(1)]
            return tgt_pred

# 采样函数
def sample_top_k(logits, k=5):
    topk_probs, topk_indices = torch.topk(torch.softmax(logits, dim=-1), k)
    idx = torch.multinomial(topk_probs, 1)
    return topk_indices.gather(-1, idx)

# 1. 加载训练好的最优模型
print("\n" + "="*60)
print("开始加载最优模型并准备翻译测试...")

# 词典路径（请确保与训练时的保存路径一致）
src_vocab_file = "D:/homework/homework/Summer/files/vocab_zh.pt"  # 源语言：中文词典
tgt_vocab_file = "D:/homework/homework/Summer/files/vocab_en.pt"  # 目标语言：英文词典
best_model_path = "D:/homework/homework/Summer/files/zh2en_model_best.pt"  # 中译英模型路径

# 加载词典
src_vocab = torch.load(src_vocab_file)
tgt_vocab = torch.load(tgt_vocab_file)
print(f"✅ 成功加载中文词典，大小：{len(src_vocab)}")
print(f"✅ 成功加载英文词典，大小：{len(tgt_vocab)}")

# 重新初始化模型结构
infer_model = TranslationModel(d_model=200, src_vocab=src_vocab, tgt_vocab=tgt_vocab)
# 加载最优模型参数
checkpoint = torch.load(best_model_path, map_location=device)
infer_model.load_state_dict(checkpoint['model_state_dict'])
# 切换为评估模式并移到指定设备
infer_model = infer_model.eval().to(device)
print(f"✅ 最优模型加载完成，当前模式：eval，设备：{device}")


# 2. 定义中译英翻译函数
def translate(src: str):
    """
    :param src: 中文句子，例如 "一位父亲和儿子坐在咖啡馆里"
    :return: 翻译后的英文句子，例如："a father and son sitting in a cafe"
    """
    # 中文句子分词 → 转词典index → 增加<BOS>和<EOS>
    src_tok = zh_tokenizer(src)
    src_indices = [0] + src_vocab(src_tok) + [1]  # 拼接特殊符号
    # 转为tensor并增加batch维度
    src_tensor = torch.tensor(src_indices).unsqueeze(0).to(device)
    
    # 初始化目标语言输入：仅包含<BOS>
    tgt_tensor = torch.tensor([[0]]).to(device)
    
    # 逐词预测
    max_gen_len = min(72, len(src_indices) )
    for _ in range(max_gen_len):
        with torch.no_grad():
            out = infer_model(src_tensor, tgt_tensor)
            pred_logits = infer_model.predictor(out[:, -1])
            pred_token_idx = sample_top_k(pred_logits, k=5).squeeze(1)

        tgt_tensor = torch.concat([tgt_tensor, pred_token_idx.unsqueeze(0)], dim=1)
        
        # 若预测到<EOS>，停止生成
        if pred_token_idx.item() == 1:
            break
    
    # 将预测的token索引转为英文文本
    tgt_tokens = tgt_vocab.lookup_tokens(tgt_tensor.squeeze().tolist())
    # 移除特殊符号并拼接英文单词（用空格分隔）
    translated_text = ' '.join([tok for tok in tgt_tokens 
                               if tok not in ["<s>", "</s>", "<pad>"]])
    return translated_text


# 3. 测试翻译功能
print("\n开始翻译测试：")
test_sentence = "一位父亲和儿子坐在咖啡馆里"
translated_result = translate(test_sentence)
print(f"中文原文：{test_sentence}")
print(f"英文译文：{translated_result}")
print("="*60)
    

当前使用设备: cuda:0

开始加载最优模型并准备翻译测试...
✅ 成功加载中文词典，大小：8280
✅ 成功加载英文词典，大小：27584
✅ 最优模型加载完成，当前模式：eval，设备：cuda:0

开始翻译测试：
中文原文：一位父亲和儿子坐在咖啡馆里
英文译文：a father and his son sat in the cafe in a cafe where a father
